# Task 2: Exploration & Basic Statistics
**Dataset:** IMDB Dataset of 50K Movie Reviews (Kaggle, lakshmi25npathi)
**Member 1:** Maham Iqbal (L1F23BSCS0411)

This notebook explores the **raw, uncleaned** reviews. All the functions live in `src/data_analysis.py`, so the notebook and the script always give the same numbers.

> Run every cell from top to bottom: *Kernel → Restart & Run All*. Then take screenshots of the outputs for the report.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Make the src/ folder importable (the notebook lives in notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import data_analysis as da
from load_data import load_raw

pd.set_option("display.max_colwidth", 200)
%matplotlib inline

## 1. Load the data and understand the columns

In [ ]:
df = load_raw()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
info = da.basic_info(df)
info

**Columns:**
- `review`: the raw review text (contains HTML tags like `<br />`)
- `sentiment`: the label, `positive` or `negative`, which is our **prediction target**

**Duplicates:** some reviews appear more than once. We report them here; the cleaning pipeline (Member 5) removes them.

In [ ]:
# Look at a few duplicated reviews
dups = df[df.duplicated(keep=False)].sort_values("review")
print("Duplicate rows (excluding first occurrence):", df.duplicated().sum())
dups.head(6)

## 2. Number of documents & average length (in words)

In [ ]:
lengths = da.doc_lengths(df)
print("Number of documents:", len(df))
da.length_stats(lengths)

In [ ]:
da.plot_length_hist(lengths, da.FIG_DIR / "length_distribution.png")
plt.show()

## 3. Target distribution

In [ ]:
da.target_distribution(df)

In [ ]:
da.plot_target(df, da.FIG_DIR / "target_distribution.png")
plt.show()

## 4. Vocabulary size (whole corpus)
Tokenizer: lowercase + regex `\w+` (`da.simple_tokenize`). This is the **baseline before cleaning**. Member 4 should use the same function for the "before" vocabulary count.

In [ ]:
print(da.simple_tokenize("This movie was GREAT!!<br /><br />I didn't expect it."))

In [ ]:
counter = da.word_counts(df["review"])
da.vocab_stats(counter)

## 5. Tokens vs types on a random sample (1000 reviews, `random_state=42`)

In [ ]:
da.sample_tokens_types(df)

In [ ]:
growth = da.type_growth(df)
da.plot_type_growth(growth, da.FIG_DIR / "type_growth.png")
plt.show()

## 6. Zipf's law and the long tail
**Zipf's law:** the frequency of a word is roughly inversely proportional to its rank (f ∝ 1/rank). On a log-log plot, this shows up as a roughly straight line sloping downwards.

In [ ]:
pd.DataFrame(counter.most_common(20), columns=["word", "frequency"]).rename_axis("rank").rename(index=lambda i: i + 1)

In [ ]:
da.plot_zipf(counter, da.FIG_DIR / "zipf_plot.png")
plt.show()

In [ ]:
da.plot_top_words(counter, 20, da.FIG_DIR / "top20_words.png")
plt.show()

In [ ]:
v = da.vocab_stats(counter)
print(f"Words that appear only ONCE (hapax legomena): {v['hapax_legomena']:,} "
      f"= {v['hapax_percent_of_vocab']}% of the vocabulary")
print(f"The top 100 words cover {v['top100_words_percent_of_tokens']}% of all tokens")

## 7. Messy text examples

In [ ]:
messy = da.find_messy_examples(df)
messy[["issue", "reviews_affected", "percent_of_reviews", "example_row", "matched_text"]]

In [ ]:
# Print the snippet for each issue (pick 3-5 of these for the report)
for _, r in messy.iterrows():
    if r["reviews_affected"]:
        print(f"### {r['issue']} | row {int(r['example_row'])} | {df.at[int(r['example_row']), 'sentiment']}")
        print(r["snippet"], "\n")

In [ ]:
# Look at any full review by its row number
row = int(messy.loc[0, "example_row"])
print(df.at[row, "review"])

## 8. Save everything for the report

In [ ]:
da.main()   # re-runs the full analysis and writes report/figures, report/stats.json, report/messy_examples.txt

## Observations (write these in your own words after running)
- **Size & duplicates:** ...
- **Length:** ... (note: whitespace word counts include HTML fragments like `<br`)
- **Target:** ... (balanced? why that matters for Phase 2)
- **Vocabulary / tokens vs types:** ...
- **Zipf & long tail:** ... (is it close to a straight line? which words are at the top? why does `br` appear?)
- **Messy text:** ... (which issues need cleaning, and which might carry sentiment, e.g. ALL CAPS, `!!!`)